# ASI example

One ASI batch: data -> fixed-model predictions -> select rows -> label -> inference.

Run from `examples/simulation/`, so that `dataset` and `model` import directly.

In [1]:
from model.judge import JudgeModel
from openppi import Mean, get_method, select_for_labeling
from dataset.judge_sim import make_judge_data

In [2]:
# 1. Separate features and labels from the data provider's DataFrame.
data = make_judge_data(n=2000, seed=0)
features = data.drop(columns='label')
judge_cols = [i for i, name in enumerate(features.columns) if name.startswith('judge')]
features = features.to_numpy()
y = data['label'].to_numpy()

In [3]:
# 2. Select the target and predict with a fixed model (any already trained model works).
target = Mean()
recipe = get_method('ASI')          # sampling: Uncertainty(tau=.5); inference: IPW
model = JudgeModel(columns=judge_cols)
predictions = model.predict(features)

In [4]:
# 3. Select rows to label from predictions only; no label is read here.
request = select_for_labeling(predictions, target=target, design=recipe.sampling(),
                              budget=400, random_state=42)
print('Rows to label:', len(request.indices))

Rows to label: 406


In [5]:
# 4. Human labeling. The simulation looks up y; in practice, export request.indices to annotators.
labels = y[request.indices]

In [6]:
# 5. Attach labels and run inference on the recorded selection.
record = request.with_labels(labels)
result = recipe.inference(record, target=target, alpha=.05)
print('Estimate:', result.estimate)
print('Confidence interval:', result.confidence_interval)
print('Diagnostics:', result.diagnostics)

Estimate: 0.50060841474748
Confidence interval: (0.451683981346233, 0.5495328481487269)
Diagnostics: {'audits': 406, 'sum_sampling_probabilities': 400.00000000000006, 'min_pi': 0.10624507259813676, 'max_pi': 0.24981901548168137, 'prediction_weight': 1.0, 'weight_strategy': 'fixed', 'variance_design': 'active_terminal_plugin'}
